# scTASL tutorial: intra-omics imputation and cross-omics translation

This notebook trains the scTASL imputation/translation model on paired
scRNA-seq and scATAC-seq data. It produces:

- denoised RNA and ATAC reconstructions (imputation),
- RNA profiles translated from ATAC cells,
- ATAC profiles translated from RNA cells,
- UMAP figures and cell-type preservation metrics for all six representations.

**Expected inputs:** `dataset/<DATASET>/rna_processed.h5ad` and
`dataset/<DATASET>/atac_processed.h5ad`, each with a `counts` layer. This
tutorial dataset also includes `obs["cell_type"]` for labeled UMAPs and
cell-type-based evaluation metrics. In real applications, cell-type labels are
not required for imputation or translation training. The current model is
designed for paired cells in matching order.

> Run this notebook from the repository root.


## 1. Setup

Explicit imports keep the notebook self-contained and make every metric and
plotting dependency visible to readers.


In [ ]:
from pathlib import Path

import anndata
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import torch
from IPython.display import display
import warnings
warnings.filterwarnings("ignore")

from model.imputation_translation_model import ImputationTranslationModel
from utils.common.config import lsi
from utils.common.data_configure import configure_dataset, load_omics_inputs
from utils.common.metrics import (
    adjusted_rand_index,
    avg_silhouette_width_label,
    compute_bcs,
    norm_lisi_label,
    normalized_mutual_information,
)
from utils.common.plots import cell_type_palette

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
sc.set_figure_params(dpi=80, facecolor="white")


### Configure the run

Set `TRAIN_MODEL=False` to reuse `Translation_final_model.pt` from the same
result directory. The three epoch values correspond to RNA warm-up, ATAC
warm-up, and latent cross-omics mapping.


In [ ]:
DATASET = "PBMC-10k"
TASK = "translation"
TRAIN_MODEL = True
RANDOM_SEED = 0

WARMUP_RNA_EPOCHS = 50
WARMUP_ATAC_EPOCHS = 50
MAPPING_EPOCHS = 200
INFERENCE_BATCH_SIZE = 256
N_COMPONENTS = 50

DATA_DIR = Path("dataset") / DATASET
RESULT_DIR = Path("results") / DATASET / TASK
ADATA_DIR = RESULT_DIR / "adata"
FIGURE_DIR = RESULT_DIR / "figure"
METRICS_DIR = RESULT_DIR / "metrics"
CHECKPOINT_PATH = RESULT_DIR / "Translation_final_model.pt"

for output_dir in (RESULT_DIR, ADATA_DIR, FIGURE_DIR, METRICS_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print(f"Dataset: {DATASET}")
print(f"Results: {RESULT_DIR.resolve()}")
print(f"Mode: {'train a new model' if TRAIN_MODEL else 'load checkpoint'}")
print(f"Compute device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")


## 2. Load and validate paired inputs

Validation is performed before model construction so missing layers, labels,
duplicate identifiers, or mismatched paired-cell order produce a clear error.


In [ ]:
# Translation training requires paired RNA/ATAC cells and count layers.
# This tutorial requires cell_type only for labeled plots and evaluation metrics.
# For unlabeled real data, set require_cell_type=False and skip label-based sections.
rna, atac, _, input_summary = load_omics_inputs(
    rna_path=DATA_DIR / "rna_processed.h5ad",
    atac_path=DATA_DIR / "atac_processed.h5ad",
    require_counts=True,
    require_cell_type=True,
    require_paired=True,
    paired_task="paired translation training",
)

display(input_summary)
print("rna:", rna, "\natac:", atac)


## 3. Prepare data loaders

The model consumes raw counts in `X`. `configure_dataset` records feature,
batch, and observation metadata required by the project data loader.


In [ ]:
rna.X = rna.layers["counts"].copy()
atac.X = atac.layers["counts"].copy()
rna.obs["cell_type"] = rna.obs["cell_type"].astype("category")
atac.obs["cell_type"] = atac.obs["cell_type"].astype("category")

adatas = {"RNA": rna, "ATAC": atac}
for adata_obj in adatas.values():
    configure_dataset(adata_obj, use_obs_names=True)

data_config = {
    omics_name: adata_obj.uns["data_config"]
    for omics_name, adata_obj in adatas.items()
}
train_loader, val_loader = ImputationTranslationModel.data_processing(
    adatas=adatas
)


## 4. Train or restore the model

Training proceeds through two unimodal VAE warm-up stages followed by the
cross-omics mapping stage. When loading, use only a checkpoint you trust;
PyTorch checkpoints rely on pickle-based serialization.


In [ ]:
model = ImputationTranslationModel(
    data_config=data_config,
    result_path=str(RESULT_DIR),
)

if TRAIN_MODEL:
    model.train_model(
        train_loader=train_loader,
        val_loader=val_loader,
        warmup_rna_epochs=WARMUP_RNA_EPOCHS,
        warmup_atac_epochs=WARMUP_ATAC_EPOCHS,
        map_epochs=MAPPING_EPOCHS,
        print_every=10,
        save_best=True,
    )
else:
    if not CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")
    state_dict = torch.load(CHECKPOINT_PATH, map_location=model.device)
    model.load_state_dict(state_dict)
    model.to(model.device)
    model.eval()
    print(f"Loaded checkpoint: {CHECKPOINT_PATH}")


## 5. Generate imputed and translated profiles

Imputation reconstructs each omics layer through its own VAE. Translation maps
the latent representation to the other omics layer before decoding. Posterior
means are used by default, making inference deterministic.


In [ ]:
rna_imputed_values = model.impute_adata(
    adata=adatas["RNA"],
    modality="RNA",
    batch_size=INFERENCE_BATCH_SIZE,
    use_mean_latent=True,
    return_numpy=True,
)
atac_imputed_values = model.impute_adata(
    adata=adatas["ATAC"],
    modality="ATAC",
    batch_size=INFERENCE_BATCH_SIZE,
    use_mean_latent=True,
    return_numpy=True,
)
rna_translated_values = model.translate_adata(
    adata=adatas["ATAC"],
    direction="ATAC2RNA",
    batch_size=INFERENCE_BATCH_SIZE,
    use_mean_latent=True,
    return_numpy=True,
)
atac_translated_values = model.translate_adata(
    adata=adatas["RNA"],
    direction="RNA2ATAC",
    batch_size=INFERENCE_BATCH_SIZE,
    use_mean_latent=True,
    return_numpy=True,
)

expected_shapes = {
    "RNA imputed": rna.shape,
    "ATAC imputed": atac.shape,
    "RNA translated from ATAC": (atac.n_obs, rna.n_vars),
    "ATAC translated from RNA": (rna.n_obs, atac.n_vars),
}
actual_shapes = {
    "RNA imputed": rna_imputed_values.shape,
    "ATAC imputed": atac_imputed_values.shape,
    "RNA translated from ATAC": rna_translated_values.shape,
    "ATAC translated from RNA": atac_translated_values.shape,
}
if actual_shapes != expected_shapes:
    raise ValueError(
        f"Unexpected model output shapes: {actual_shapes}; "
        f"expected {expected_shapes}."
    )
display(pd.DataFrame({"shape": actual_shapes}))


## 6. Construct analysis-ready AnnData objects

The model outputs are continuous reconstructed means rather than observed
integer counts. They are stored in both `X` and `layers["model_output"]` to
make that distinction explicit. RNA uses PCA; ATAC uses latent semantic
indexing (LSI) before neighbor graph and UMAP construction.


In [ ]:
def build_model_output_adata(values, obs, var):
    # Preserve cell and feature annotations in each model-derived object.
    output = anndata.AnnData(
        X=values.copy(),
        obs=obs.copy(),
        var=var.copy(),
    )
    output.layers["model_output"] = values.copy()
    return output

rna_imputed = build_model_output_adata(
    rna_imputed_values, rna.obs, rna.var
)
atac_imputed = build_model_output_adata(
    atac_imputed_values, atac.obs, atac.var
)
rna_translated = build_model_output_adata(
    rna_translated_values, atac.obs, rna.var
)
atac_translated = build_model_output_adata(
    atac_translated_values, rna.obs, atac.var
)

representations = {
    "rna_imputed": rna_imputed,
    "rna_translated": rna_translated,
    "atac_imputed": atac_imputed,
    "atac_translated": atac_translated,
}


In [ ]:
def prepare_rna_representation(adata_obj, random_state):
    # Normalize RNA-like values and calculate PCA, neighbors, and UMAP.
    sc.pp.normalize_total(adata_obj)
    sc.pp.log1p(adata_obj)
    n_components = min(N_COMPONENTS, adata_obj.n_obs - 1, adata_obj.n_vars - 1)
    if n_components < 2:
        raise ValueError("RNA PCA requires at least three cells and features.")
    sc.tl.pca(adata_obj, n_comps=n_components)
    sc.pp.neighbors(
        adata_obj,
        n_neighbors=min(20, adata_obj.n_obs - 1),
        n_pcs=n_components,
    )
    sc.tl.umap(adata_obj, random_state=random_state)


def prepare_atac_representation(adata_obj, random_state):
    # Calculate ATAC LSI, cosine neighbors, and UMAP.
    n_components = min(N_COMPONENTS, adata_obj.n_obs - 1, adata_obj.n_vars - 1)
    if n_components < 2:
        raise ValueError("ATAC LSI requires at least three cells and features.")
    lsi(adata_obj, n_components=n_components, n_iter=15)
    sc.pp.neighbors(adata_obj, use_rep="X_lsi", metric="cosine")
    sc.tl.umap(adata_obj, random_state=random_state)


for name in ("rna_imputed", "rna_translated"):
    prepare_rna_representation(representations[name], RANDOM_SEED)
for name in ("atac_imputed", "atac_translated"):
    prepare_atac_representation(representations[name], RANDOM_SEED)


## 7. Save processed representations

Internal `data_config` metadata is removed only from output copies, preserving
the in-memory objects and model configuration for the rest of the notebook.


In [ ]:
adata_output_paths = {
    name: ADATA_DIR / f"{name}.h5ad" for name in representations
}
for name, adata_obj in representations.items():
    output_adata = adata_obj.copy()
    output_adata.uns.pop("data_config", None)
    output_adata.write(adata_output_paths[name], compression="gzip")
    print(f"Saved {name}: {adata_output_paths[name]}")


## 8. Visualize cell-type structure

Each panel uses the same label-aware palette. These plots show whether
imputation or translation preserves recognizable biological neighborhoods;
they do not by themselves measure reconstruction accuracy.


In [ ]:
for name, adata_obj in representations.items():
    figure = sc.pl.umap(
        adata_obj,
        color="cell_type",
        palette=cell_type_palette(adata_obj),
        title=name.replace("_", " ").title(),
        show=False,
        return_fig=True,
    )
    figure_path = FIGURE_DIR / f"{name}_umap_cell_type.pdf"
    figure.savefig(figure_path, bbox_inches="tight")
    display(figure)
    plt.close(figure)


## 9. Evaluate cell-type preservation

NMI and ARI compare clusters with known cell types. ASW label, BCS, and cLISI
quantify cell-type structure in PCA or LSI space. Higher normalized values
indicate stronger preservation of cell-type structure.


In [ ]:
def calculate_cell_type_preservation_scores(embedding, labels):
    # Calculate five cell-type preservation metrics and their mean.
    scores = {
        "NMI": float(normalized_mutual_information(embedding, labels)),
        "ARI": float(adjusted_rand_index(embedding, labels)),
        "ASW label": float(avg_silhouette_width_label(embedding, labels)),
        "BC": float(compute_bcs(embedding, labels)),
        "cLISI": float(norm_lisi_label(embedding, labels)),
    }
    scores["Mean score"] = float(
        np.mean(list(scores.values()))
    )
    return scores


metric_rows = []
for name, adata_obj in representations.items():
    embedding_key = "X_pca" if name.startswith("rna_") else "X_lsi"
    scores = calculate_cell_type_preservation_scores(
        np.asarray(adata_obj.obsm[embedding_key]),
        adata_obj.obs["cell_type"],
    )
    metric_rows.append({"representation": name, **scores})

metrics_table = pd.DataFrame(metric_rows).set_index("representation")
display(metrics_table.style.format("{:.4f}"))

metrics_path = METRICS_DIR / "cell_type_preservation_scores.csv"
metrics_table.to_csv(metrics_path)
print(f"Saved metrics: {metrics_path}")


## 10. Next steps

Use the saved H5AD files for downstream differential, marker, or cross-omics
analyses. Cell-type preservation metrics should be interpreted together with
feature-level reconstruction or translation metrics appropriate to the
dataset. To reuse a trained model, keep the same data features and set
`TRAIN_MODEL=False` before rerunning from the top.
